<a href="https://colab.research.google.com/github/SkyWarLimit/PemMes_17/blob/main/JS09/JS09_Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -q faiss-cpu annoy hnswlib

import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# 1. Baca header dataset terlebih dahulu untuk memverifikasi kolom yang tersedia
print("Memeriksa kolom yang tersedia di dataset...")
temp_df = pd.read_csv('songs_with_attributes_and_lyrics.csv', nrows=0)
available_cols = list(temp_df.columns)
print("Kolom yang ada di file:", available_cols)

# Menentukan target fitur
target_features = ['danceability', 'energy', 'loudness', 'speechiness',
                   'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Filter fitur yang benar-benar ada di file
features = [col for col in target_features if col in available_cols]
if not features:
    raise ValueError("Tidak ada fitur numerik target yang ditemukan di dalam CSV!")

# Cek apakah 'track_name' atau alternatif kolom string nama ada
name_col = None
for col_candidate in ['track_name', 'name', 'title', 'track']:
    if col_candidate in available_cols:
        name_col = col_candidate
        break

cols_to_load = features.copy()
if name_col:
    cols_to_load.append(name_col)

# 2. Load dataset menggunakan python engine agar lebih toleran terhadap bad lines dan quotes bermasalah
print(f"Memuat dataset dengan kolom: {cols_to_load}...")
try:
    df = pd.read_csv('songs_with_attributes_and_lyrics.csv', usecols=cols_to_load, engine='python', on_bad_lines='skip')
except Exception as e:
    print("Terjadi error, mencoba membaca ulang dengan menonaktifkan pemrosesan tanda kutip (quoting)...")
    import csv
    df = pd.read_csv('songs_with_attributes_and_lyrics.csv', usecols=cols_to_load, engine='python', quoting=csv.QUOTE_NONE, on_bad_lines='skip')

# 3. Buang baris yang memiliki nilai kosong
df = df.dropna(subset=features).reset_index(drop=True)

# 4. Ambil nilai numeriknya
X = df[features].values

# 5. Standarisasi dan ubah ke tipe data float32
print("Melakukan standarisasi data...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10  # jumlah nearest neighbors
query_idx = 0
query_vector = X_scaled[query_idx]

print(f"Total data yang siap diproses: {X_scaled.shape[0]} lagu.")
track_identifier = df.iloc[query_idx].get(name_col, 'Unknown') if name_col else f"Indeks-{query_idx}"
print(f"Mencari lagu yang mirip dengan: {track_identifier}")

Memeriksa kolom yang tersedia di dataset...
Kolom yang ada di file: ['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']
Memuat dataset dengan kolom: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'name']...
Melakukan standarisasi data...
Total data yang siap diproses: 253027 lagu.
Mencari lagu yang mirip dengan: !


In [14]:
print("--- Exact Nearest Neighbors (Sklearn) ---")
start_time = time.time()

# Menggunakan NearestNeighbors bawaan sklearn
exact_nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
exact_nn.fit(X_scaled)
distances_exact, indices_exact = exact_nn.kneighbors([query_vector])

exact_time = time.time() - start_time
print(f"Waktu Eksekusi : {exact_time:.6f} detik")
print(f"Indeks hasil   : {indices_exact[0]}")

--- Exact Nearest Neighbors (Sklearn) ---
Waktu Eksekusi : 0.101124 detik
Indeks hasil   : [     0  61511  85956 240692 216265   3836 142612  35205 209597 192567]


In [15]:
print("--- Approximate Nearest Neighbors (ANNOY) ---")
start_time = time.time()

d = X_scaled.shape[1]
annoy_index = AnnoyIndex(d, 'euclidean')

# Build index
for i in range(X_scaled.shape[0]):
    annoy_index.add_item(i, X_scaled[i])
annoy_index.build(10) # 10 n_trees

# Pencarian
indices_annoy = annoy_index.get_nns_by_vector(query_vector, k)

annoy_time = time.time() - start_time
print(f"Waktu Eksekusi : {annoy_time:.6f} detik")
print(f"Indeks hasil   : {indices_annoy}")

--- Approximate Nearest Neighbors (ANNOY) ---
Waktu Eksekusi : 8.559537 detik
Indeks hasil   : [0, 61511, 85956, 240692, 216265, 3836, 35205, 209597, 192567, 41311]


In [16]:
print("--- Approximate Nearest Neighbors (FAISS) ---")
start_time = time.time()

d = X_scaled.shape[1]
faiss_index = faiss.IndexFlatL2(d)

# Build index & Pencarian
faiss_index.add(X_scaled)
distances_faiss, indices_faiss = faiss_index.search(np.array([query_vector]), k)

faiss_time = time.time() - start_time
print(f"Waktu Eksekusi : {faiss_time:.6f} detik")
print(f"Indeks hasil   : {indices_faiss[0]}")

--- Approximate Nearest Neighbors (FAISS) ---
Waktu Eksekusi : 0.101658 detik
Indeks hasil   : [     0  61511  85956 240692 216265   3836 142612  35205 209597 192567]


In [17]:
print("--- Approximate Nearest Neighbors (HNSW) ---")
start_time = time.time()

d = X_scaled.shape[1]
hnsw_index = hnswlib.Index(space='l2', dim=d)

# Build index & Pencarian
hnsw_index.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
hnsw_index.set_ef(50)

indices_hnsw, distances_hnsw = hnsw_index.knn_query(np.array([query_vector]), k=k)

hnsw_time = time.time() - start_time
print(f"Waktu Eksekusi : {hnsw_time:.6f} detik")
print(f"Indeks hasil   : {indices_hnsw[0]}")

--- Approximate Nearest Neighbors (HNSW) ---
Waktu Eksekusi : 53.100078 detik
Indeks hasil   : [     0  61511  85956 240692 216265   3836 142612  35205 209597 192567]


In [19]:
# Membandingkan kesamaan indeks yang ditemukan oleh Exact NN dengan ANN
exact_set = set(indices_exact[0])

recall_annoy = len(exact_set.intersection(indices_annoy)) / k
recall_faiss = len(exact_set.intersection(indices_faiss[0])) / k
recall_hnsw = len(exact_set.intersection(indices_hnsw[0])) / k

print("=== Kesimpulan Perbandingan Algoritma ===")
results_df = pd.DataFrame({
    'Algorithm': ['Exact NN', 'ANNOY', 'FAISS', 'HNSW'],
    'Time (Seconds)': [exact_time, annoy_time, faiss_time, hnsw_time],
    'Recall (vs Exact)': ['100%', f"{recall_annoy*100}%", f"{recall_faiss*100}%", f"{recall_hnsw*100}%"]
})

display(results_df)


=== Kesimpulan Perbandingan Algoritma ===


,Algorithm,Time (Seconds),Recall (vs Exact)
0,Exact NN,0.101124,100%
1,ANNOY,8.559537,90.0%
2,FAISS,0.101658,100.0%
3,HNSW,53.100078,100.0%
